In [1]:
from functions import ping_openai, fetch_articles, read_prompt_from_file_only, load_function_schema, normalize_id, normalize_type, combine_paragraphs, format_nodes_for_prompt, get_schema, pretty_print_json
from model_dictionary import model_dictionary
from inputs import relationship_groups, groups_to_prompts, nodes_by_group_prompt, characteristic_node_types

import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from openai import OpenAI
from dotenv import load_dotenv
import json
from bson import json_util
import re
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
ARTICLES_COLLECTION_NAME = os.getenv("MONGO_COLLECTION_NAME")

# Ensure all necessary environment variables are set
if not all([MONGO_URI, DB_NAME, ARTICLES_COLLECTION_NAME]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()

# Create MongoDB client with TLS verification
try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]

    # Define collections
    articles_collection = db[ARTICLES_COLLECTION_NAME]  # Stores articles

    # Verify connection
    client.admin.command('ping')
    print(f"✅ Connected to MongoDB Atlas! Database: {DB_NAME}")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}")
    exit()

openAI_api_key = os.getenv('OPENAI_API_KEY')
client = OpenAI(api_key=openAI_api_key)
    
ping_openai(client)

# DO WE NEED THIS FUNCTION STILL?
#articles = fetch_articles(articles_collection, limit=1,offset=15)

✅ Connected to MongoDB Atlas! Database: tuone
✅ Successfully connected to OpenAI API!
Available Models: ['gpt-4o-audio-preview-2024-12-17', 'dall-e-3', 'dall-e-2', 'gpt-4o-audio-preview-2024-10-01', 'text-embedding-3-small', 'o4-mini', 'gpt-4.1-nano', 'gpt-4.1-nano-2025-04-14', 'gpt-4o-realtime-preview-2024-10-01', 'o4-mini-2025-04-16', 'gpt-4o-realtime-preview', 'babbage-002', 'gpt-4', 'text-embedding-ada-002', 'text-embedding-3-large', 'gpt-4o-mini-audio-preview', 'gpt-4o-audio-preview', 'o1-preview-2024-09-12', 'gpt-4o-mini-realtime-preview', 'gpt-4.1-mini', 'gpt-4o-mini-realtime-preview-2024-12-17', 'gpt-3.5-turbo-instruct-0914', 'gpt-4o-mini-search-preview', 'gpt-4.1-mini-2025-04-14', 'computer-use-preview-2025-03-11', 'chatgpt-4o-latest', 'davinci-002', 'gpt-3.5-turbo-1106', 'gpt-4o-search-preview', 'gpt-4-turbo', 'gpt-4o-realtime-preview-2024-12-17', 'gpt-3.5-turbo-instruct', 'gpt-3.5-turbo', 'gpt-4-turbo-preview', 'gpt-4o-mini-search-preview-2025-03-11', 'gpt-4-0125-preview', '

In [2]:
def extract_nodes(article_id, text, PROMPT_PATH, FUNCTION_SCHEMA_PATH, model_name):

    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    try:
        completion = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": f"Here is the article: {text}"}
            ],
            functions = [function_schema],
            function_call = {"name": "extract_clean_tech_entities"},
            temperature=0
        )

        # parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments

        #an_example_output_for_finetuning = {
        #    "messages": [
        #        {"role": "system", "content": prompt},
        #        {"role": "user", "content": f"Here is the article: {text}"},
        #        {"role": "assistant", "content": function_args}
        #    ]
        #}

        #with open("example_for_finetuning.json", "w", encoding="utf-8") as f:
        #    json.dump(example_for_finetuning, f, ensure_ascii=False, indent=2)

        extracted_data = json.loads(function_args)
        print(f"✅ Succesful call to OpenAI. The following nodes are returned:")
        pretty_print_json(extracted_data, "Nodes returned by OpenAI")

        if "nodes" not in extracted_data:
            raise ValueError("❌ Expected 'nodes' key in LLM output but not found.")

        nodes_by_category = extracted_data["nodes"]

        # Flatten nodes while retaining their type
        formatted_nodes = []
        for node_type, node_categories in nodes_by_category.items():
            for node in node_categories:
                formatted_node = {
                    "article_id": article_id,
                    "id": normalize_id(node.get("id")),
                    "type": normalize_type(node.get("type")),
                    "name": node.get("name"),
                    "location": {
                        "city": node.get("location", {}).get("city", ""),
                        "country": node.get("location", {}).get("country", "")
                    } if node.get("location") else None,
                    "amount": node.get("amount"), #possibly to drop
                    "status": node.get("status"),
                }
                formatted_nodes.append(formatted_node)

        return formatted_nodes

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error: {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting nodes: {e}")
        return [] 

In [3]:
def extract_node_characteristics(article_id, text, nodes, relationship_group, model_name):

    try:
        PROMPT_PATH = groups_to_prompts[relationship_group]
    except KeyError:
        print(f"❌ PROMPT_PATH missing for relationship_group '{relationship_group}'")
        return []

    try:
        FUNCTION_SCHEMA_PATH = f"schemas/{relationship_group}.json"
        function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    except FileNotFoundError:
        print(f"❌ Schema file not found for '{relationship_group}'")
        return []

    try:
        allowed_types = nodes_by_group_prompt[relationship_group]
    except KeyError:
        print(f"❌ allowed_types missing for relationship_group '{relationship_group}'")
        return []

    PROMPT_PATH = groups_to_prompts[relationship_group]
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    FUNCTION_SCHEMA_PATH = f"schemas/{relationship_group}.json"
    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)

    allowed_types = nodes_by_group_prompt[relationship_group]
    compact_nodes = format_nodes_for_prompt(nodes, allowed_types)
    
    try:
        completion = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": prompt},
                {
                "role": "user",
                 "content": f"""Here is the article text: {text}

                Here is the list of production capacities:
                {compact_nodes}
                """
                }
            ],
            functions=[function_schema],
            function_call={"name": "extract_characteristics"},
            temperature=0
        )

        #parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments
        extracted_data = json.loads(function_args)
        print(f"✅ Succesful openAI call: returned the following capacity characteristics:")
        pretty_print_json(extracted_data, "Relationships returned by OpenAI")

        if "node_characteristics" not in extracted_data:
            raise ValueError("Expected 'node_characteristics' key in LLM output but not found.")

        return extracted_data["node_characteristics"]

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error (relationships): {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting relationships: {e}")
        return [] 

In [4]:
def extract_relationships(article_id, text, nodes, relationship_group, model_name, allowed_types=None):

    PROMPT_PATH = groups_to_prompts[relationship_group]
    prompt = read_prompt_from_file_only(PROMPT_PATH)

    FUNCTION_SCHEMA_PATH = f"schemas/{relationship_group}.json"
    function_schema = load_function_schema(FUNCTION_SCHEMA_PATH)
    
    allowed_types = nodes_by_group_prompt[relationship_group]
    compact_nodes = format_nodes_for_prompt(nodes, allowed_types)
    
    try:
        completion = client.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": prompt},
                {
                "role": "user",
                 "content": f"""Here is the article text: {text}

                Here is the list of known entities:
                {compact_nodes}

                Please extract only the specified relationship types."""
                }
            ],
            functions=[function_schema],
            function_call={"name": "extract_clean_tech_relationships"},
            temperature=0
        )

        #parse the returned function call
        function_args = completion.choices[0].message.function_call.arguments
        extracted_data = json.loads(function_args)
        print(f"✅ Succesful openAI call: returned the following relationships:")
        pretty_print_json(extracted_data, "Relationships returned by OpenAI")

        if "relationships" not in extracted_data:
            raise ValueError("Expected 'relationships' key in LLM output but not found.")

        formatted_relationships = []
        for rel in extracted_data["relationships"]:
            raw_source = rel.get("source")
            raw_target = rel.get("target")
            norm_source = normalize_id(raw_source)
            norm_target = normalize_id(raw_target)

            #print(f"🔍 Normalizing IDs - Source: {raw_source} → {norm_source}, Target: {raw_target} → {norm_target}")

            formatted_relationships.append({
                "article_id": article_id,
                "id": f"{norm_source}_{rel.get('type')}_{norm_target}",
                "source": norm_source,
                "target": norm_target, # should this be changed to raw_target?
                "type": rel.get("type"),
                "group": relationship_group
            })

        return formatted_relationships

    except json.JSONDecodeError as e:
        print(f"❌ JSON Parsing Error (relationships): {e}")
        return []
    except Exception as e:
        print(f"❌ Error extracting relationships: {e}")
        return []

In [5]:
def process_articles(limit, offset, PROMPT_PATH, FUNCTION_SCHEMA_PATH, model_dictionary):
    # Retrieve articles with limit & offset, sorted by `_id` for consistency
    articles_to_process = list(
        articles_collection.find(
            {"meta.ID": {"$regex": "^27"}},  
            {"_id": 1, "paragraphs": 1, "validation": 1}       
        )
        .sort("_id", -1)  # Sort by MongoDB ObjectId (descending)
        .skip(offset)    # Skip first `offset` articles
        .limit(limit)    # Limit the number of articles
    )

    print(f"🔹 Processing {len(articles_to_process)} articles (Offset: {offset})")

    for article in articles_to_process:
        articleID = str(article["_id"])  #convert ObjectId to string

        if article.get("validation") is True:
            print(f"⏭️  Skipping Article ID: {articleID} - Article is validated")
            continue

        text = combine_paragraphs(article)
        if not text:
            print(f"⚠️ No valid text found for Article ID: {articleID}. Skipping.")
            continue

        print(f"📌 Processing Article ID: {articleID}")

        try:
            formatted_nodes = extract_nodes(articleID, text, PROMPT_PATH, FUNCTION_SCHEMA_PATH, model_dictionary["nodes"])

            for relationship_group, config in characteristic_node_types.items():
                model_name = model_dictionary[relationship_group]
                id_key = config["id_key"]
                type_match = config["type_match"]

                # Only continue if there are nodes of this type in the article
                has_relevant_nodes = any(node.get("type") == type_match for node in formatted_nodes)
                if not has_relevant_nodes:
                    print(f"⏭️ Skipping {relationship_group} – no nodes of type '{type_match}' found.")
                    continue

                print(f"🔍 Extracting characteristics for node group: {relationship_group}")
                node_characteristics = extract_node_characteristics(articleID, text, formatted_nodes, relationship_group, model_name)  

                # Directly attach 'status' and 'type' to matching capacity nodes (flat list structure)
                for char in node_characteristics:
                    node_id = char.get(id_key)
                    found_match = False

                    for node in formatted_nodes:
                        if node.get("id") == node_id and node.get("type") == type_match:
                            print(f"✅ Match found for {id_key} '{node_id}' in {type_match} nodes")
                            if "status" in char:
                                node["status"] = char["status"]
                                print(f"  ➕ Set status: {char['status']}")
                            if "phase" in char:
                                node["phase"] = char["phase"]
                                print(f"  ➕ Set phase: {char['phase']}")
                            found_match = True

                    if not found_match:
                        print(f"⚠️ No match found for {id_key} '{node_id}' in formatted_nodes")

            all_relationships = []
            for relationship_group,model_name in model_dictionary.items():
                if relationship_group in ["nodes", "capacities", "investments"]:
                    continue

                print(f"- - - Querying openai for relationships: {relationship_group}, using model: {model_name}")

                allowed_types = relationship_groups[relationship_group]
                print(f"Node types included in the query: {allowed_types}")

                relationships = extract_relationships(articleID, text, formatted_nodes, relationship_group, model_name, allowed_types)
                all_relationships.extend(relationships)

            pretty_print_json(formatted_nodes, "Formatted nodes to be written to mongodb")
            update_result = articles_collection.update_one(
                {"_id": article["_id"]},  #match by article id
                {"$set": {
                    "nodes": formatted_nodes or [],
                    "relationships": all_relationships or []
                    }})

            if update_result.modified_count > 0:
                print(f"✅ Updated Article ID: {articleID} with {len(formatted_nodes)} nodes and combined text.")
            else:
                print(f"⚠️ No updates made for Article ID: {articleID}.")

        except Exception as e:
            print(f"❌ Error processing Article ID {articleID}: {e}")
            

In [9]:
n_articles = 100
offset_articles = 0

PROMPT_PATH = "prompts/entities-only.txt"
FUNCTION_SCHEMA_PATH = "schemas/entities.json"

process_articles(n_articles,offset_articles, PROMPT_PATH, FUNCTION_SCHEMA_PATH, model_dictionary)

🔹 Processing 100 articles (Offset: 0)
⏭️  Skipping Article ID: 67f52d07981040986eab7b6c - Article is validated
📌 Processing Article ID: 67f52d07981040986eab7b6b


KeyboardInterrupt: 

In [13]:
articles_collection.find_one({"title": "VW Electric Cars to be Built in U.S."})

{'_id': ObjectId('67fbf9b694b7f855cfea75d6'),
 'title': 'VW Electric Cars to be Built in U.S.',
 'paragraphs': [{'p1': '',
   'p2': 'The scandal of VW’s cooking the books over diesel emissions and fuel efficiency cost the company more than $15 billion in damages worldwide, most of that in the U.S. That included buybacks of nearly 500,000 affected Volkswagen vehicles and sister company Audi diesel vehicles, and brought down top executives in both companies. Not to mention the dent on its reputation.',
   'p3': 'Dieselgate also halted demand for diesel family cars like the Passat diesel and Chevy Cruze diesel in both the U.S. and Europe, although diesels remain in demand for hard-working trucks and buses, including school buses.',
   'p4': 'That’s just a little background for you to understand what is essentially a new company built on the ashes of Dieselgate, including killing off the Beetle, the most famous VW in history and the model that launched the brand just after WWII. The final 

In [7]:
offset = 3
limit = 1

articles_to_process = list(
    articles_collection.find(
        {"meta.ID": {"$regex": "^27"}},  # <-- Add this filter
        {"_id": 1, "paragraphs": 1, "validation": 1, "meta": 1, "title": 1,
         "nodes":1, "relationships":1}       # Keep projection as is
    )
    .sort("_id", -1)  # Sort by MongoDB ObjectId (descending)
    .skip(offset)    # Skip first `offset` articles
    .limit(limit)    # Limit the number of articles
)

articles_to_process

[{'_id': ObjectId('67f52d07981040986eab7b69'),
  'title': 'Honda invests in production of fuel cell drives',
  'paragraphs': [{'p1': 'Honda wants to create a production base for next-generation fuel cell systems in Japan. The facilities will be located in existing buildings in Mooka, where Honda ceased production of car drive components in October.',
    'p2': 'The Japanese car manufacturer has announced its intention to produce the next generation of its fuel cell systems in Mooka in the Japanese prefecture of Tochigi. The site and buildings of Honda’s Powertrain Unit Factory, which has not been in operation since October, will be used partly for this purpose. Honda expects FC production to start in the 2027/2028 financial year, with a target production capacity of 30,000 units per year. Honda is counting on government subsidies to finance the project, as Japan is considered very open to hydrogen mobility.',
    'p3': 'Honda is currently reorganising itself. There are even indications

Validation tracking

In [10]:
validated_count = articles_collection.count_documents({"validation": True})
print(f"Number of validated articles: {validated_count}")

# If you want to specifically count only electric vehicle articles that are validated:
ev_validated_count = articles_collection.count_documents({
    "validation": True,
    "meta.ID": {"$regex": "^27"}  # Same filter as in your process_relationships function
})
print(f"Number of validated Electrive Automobile articles: {ev_validated_count}")

# If you want to specifically count only electric vehicle articles that are validated:
ev_validated_count = articles_collection.count_documents({
    "validation": True,
    "meta.ID": {"$regex": "^33"}  # Same filter as in your process_relationships function
})
print(f"Number of validated World Energy Vehicle articles: {ev_validated_count}")

Number of validated articles: 96
Number of validated Electrive Automobile articles: 18
Number of validated World Energy Vehicle articles: 56
